# Ejecuciones y Resultados

<div style="text-align: justify;">

Este _notebook_ ejecuta las variantes del _NSGA-II_ con los hiperparámetros obtenidos en la parametrización (`Results/Parametrizacion`) sobre los datasets de evaluación (`B1`, `B2`, `B3`), usando **31 semillas** derivadas de la fecha del día y compartidas por todas las variantes. Cada ejecución se guarda de forma incremental (CSV de historial + frente F0 en `.npz`), por lo que una sesión interrumpida se reanuda sin repetir lo ya ejecutado.

</div>

In [1]:
# 1. Clonar el repo
#!git clone https://github.com/BenjaminGonzalezH/Thesis.git /content/Thesis

# 2. Montar Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 3. Copiar las matrices desde Drive al directorio que usa el notebook
import shutil
from pathlib import Path

DRIVE_MATRICES = Path("/content/drive/MyDrive/")                   # <-- ajusta a tu carpeta en Drive
DEST_MATRICES  = Path("/content/Thesis/Datasets/Matrices")        # minúscula: así lo espera el notebook en Colab

DEST_MATRICES.mkdir(parents=True, exist_ok=True)

archivos = list(DRIVE_MATRICES.glob("*.npz"))
for f in archivos:
    shutil.copy2(f, DEST_MATRICES / f.name)

print(f"{len(archivos)} archivos copiados a {DEST_MATRICES}")
print(sorted(p.name for p in DEST_MATRICES.iterdir()))

0 archivos copiados a \content\Thesis\Datasets\Matrices
['A1_S_DB.npz', 'A1_S_DE.npz', 'A2_S_DB.npz', 'A2_S_DB_M.npz', 'A2_S_DE.npz', 'B1_S_DB.npz', 'B1_S_DB_M.npz', 'B1_S_DE.npz', 'B2_S_DB.npz', 'B2_S_DB_M.npz', 'B2_S_DE.npz', 'B3_S_DB.npz', 'clustered_heatmap_BI_A1.png', 'clustered_heatmap_BI_A1_M.png', 'clustered_heatmap_BI_A2.png', 'clustered_heatmap_BI_A2_M.png', 'clustered_heatmap_BI_B1.png', 'clustered_heatmap_BI_B1_M.png', 'clustered_heatmap_BI_B2.png', 'clustered_heatmap_BI_B2_M.png', 'clustered_heatmap_BI_B3.png', 'clustered_heatmap_BI_B3_M.png', 'clustered_heatmap_GE_A1.png', 'clustered_heatmap_GE_A2.png', 'clustered_heatmap_GE_B1.png', 'clustered_heatmap_GE_B2.png', 'clustered_heatmap_GE_B3.png']


## 1. Ambiente y semillas

In [2]:
#######################################################
# MODULO DE PREPARACIÓN DEL AMBIENTE
# Detección de entorno (Colab / Local), rutas del repositorio, respaldo en
# Drive y semillas de las ejecuciones (31, derivadas de la fecha del día).
#######################################################

############################
# Importaciones
############################
import json                                                 # Guardado de las semillas y de las configuraciones.
import sys                                                  # Ajuste de sys.path para importar los módulos de Source Code.
import os                                                   # Llamadas al SO.
import shutil                                               # Copia de archivos entre el disco local y Drive.
from datetime import date                                   # Semilla basada en la fecha del día.
import numpy  as np                                         # Aplicación de operaciones matematicas eficientes.
import pandas as pd                                         # Historial y resumen de resultados.
from pathlib import Path                                    # Aplicar un formato generalizado de direcciones en el script.

############################
# Detección de entorno (Colab / Local)
############################
try:
    import google.colab                                     # Módulo disponible solo dentro de runtimes de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError(f"No se encontró la raíz del repositorio (Datasets/ y Source Code/) desde {_cwd}")

MATRICES_DIR = THESIS_DIR / "Datasets" / "Matrices"
print(f"[entorno] {'Colab' if IN_COLAB else 'Local'} → {THESIS_DIR}")

SOURCE_CODE_DIR = THESIS_DIR / "Source Code"
if str(SOURCE_CODE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_CODE_DIR))

############################
# Directorios de resultados y respaldo en Drive
############################
PARAM_DIR   = THESIS_DIR / "Results" / "Parametrizacion"   # Hiperparámetros obtenidos en la parametrización.
RESULTS_DIR = THESIS_DIR / "Results" / "Ejecuciones"       # Resultados de este notebook.
FRENTES_DIR = RESULTS_DIR / "Frentes"                      # Un .npz por ejecución (F0, medoides e historial de HV).
FRENTES_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Thesis_Results/Ejecuciones")    # <-- ajusta a tu carpeta en Drive
DRIVE_PARAM_DIR   = Path("/content/drive/MyDrive/Thesis_Results/Parametrizacion")

def _copiar_arbol(origen: Path, destino: Path, restaurar: bool) -> None:
    """Copia recursiva de `origen` a `destino`. Con `restaurar`, no pisa archivos existentes; si no, solo copia lo nuevo o modificado."""
    if not origen.exists():
        return
    for archivo in origen.rglob("*"):
        if not archivo.is_file():
            continue
        objetivo = destino / archivo.relative_to(origen)
        if objetivo.exists() and (restaurar or objetivo.stat().st_mtime >= archivo.stat().st_mtime):
            continue
        objetivo.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archivo, objetivo)

def sincronizar_drive(restaurar: bool = False) -> None:
    """Local -> Drive (o Drive -> local si `restaurar`). No hace nada fuera de Colab."""
    if not IN_COLAB:
        return
    try:
        if restaurar:
            _copiar_arbol(DRIVE_PARAM_DIR,   PARAM_DIR,   restaurar=True)
            _copiar_arbol(DRIVE_RESULTS_DIR, RESULTS_DIR, restaurar=True)
        else:
            _copiar_arbol(RESULTS_DIR, DRIVE_RESULTS_DIR, restaurar=False)
    except Exception as e:                                  # Un fallo de Drive no debe interrumpir las ejecuciones.
        print(f"[drive] no se pudo {'restaurar' if restaurar else 'respaldar'}: {e}")

sincronizar_drive(restaurar=True)

############################
# Semillas (31, derivadas de la fecha del día AAAAMMDD)
############################
N_SEEDS = 31
SEEDS_FILE = RESULTS_DIR / "seeds_ejecuciones.json"

SEED = int(date.today().strftime("%Y%m%d"))                 # Semilla del día: origen de las 31 semillas.
SEEDS = [int(s) for s in np.random.default_rng(SEED).integers(0, 2**31 - 1, size=N_SEEDS)]

# Persistencia: si ya existen semillas guardadas (sesión previa), se reutilizan para que
# la serie de ejecuciones sea siempre la misma aunque el notebook se reanude otro día.
if SEEDS_FILE.exists():
    with open(SEEDS_FILE, "r", encoding="utf-8") as f:
        _guardado = json.load(f)
    SEED, SEEDS = _guardado["seed"], _guardado["seeds"]
    print(f"[semillas] cargadas desde {SEEDS_FILE.name} (generadas el {_guardado['fecha']})")
else:
    with open(SEEDS_FILE, "w", encoding="utf-8") as f:
        json.dump({"fecha": date.today().isoformat(), "seed": SEED, "seeds": SEEDS}, f, indent=2)
    print(f"[semillas] archivo creado: {SEEDS_FILE.name}")
print(f"[semillas] SEED = {SEED} | {len(SEEDS)} semillas: {SEEDS[:5]} ...")
print(f"[resultados] {RESULTS_DIR}")

[entorno] Local → C:\Users\benja\Desktop\workspace\Thesis
[semillas] cargadas desde seeds_ejecuciones.json (generadas el 2026-11-01)
[semillas] SEED = 20261101 | 31 semillas: [1183181808, 1270388439, 1658144228, 768834435, 978234145] ...
[resultados] C:\Users\benja\Desktop\workspace\Thesis\Results\Ejecuciones


## 2. Código fuente e hiperparámetros

In [3]:
#######################################################
# MODULO DE IMPORTACIÓN DEL CÓDIGO FUENTE E HIPERPARÁMETROS
#######################################################

import solution_encoding                                    # OBJ_FUNCTION_CALLS es un contador global: se reinicia antes de cada ejecución.
from solution_encoding import load_distance_matrix
from LS import build_neighborhood_matrix                    # M_V de MOLS: GE, BI o BOTH.
from nsga2 import run_nsga2
from nsga2_variants import run_nsga2_MOPR, run_nsga2_PLS, run_nsga2_MOLS
from nsga2_variants_bio import run_nsga2_PNS, run_nsga2_DBS

VARIANTES = {
    "NSGA2":       run_nsga2,
    "NSGA2_PLS":   run_nsga2_PLS,
    "NSGA2_MOLS":  run_nsga2_MOLS,
    "NSGA2_DBS":   run_nsga2_DBS,
    "NSGA2_PNS":   run_nsga2_PNS,
    "NSGA2_MOPR":  run_nsga2_MOPR,
}

############################
# Mejores hiperparámetros de la parametrización
############################
def cargar_mejores_parametros() -> dict:
    """
    Lee `mejores_parametros.json`. Si no existe (la parametrización no llegó a escribirlo),
    se reconstruye desde los estudios de Optuna (`<variante>.db`).
    """
    archivo = PARAM_DIR / "mejores_parametros.json"
    if archivo.exists():
        with open(archivo, "r", encoding="utf-8") as f:
            return json.load(f)

    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    mejores = {}
    for variante in VARIANTES:
        db = PARAM_DIR / f"{variante}.db"
        if not db.exists():
            raise FileNotFoundError(f"Sin parámetros para {variante}: falta {archivo.name} y {db.name} en {PARAM_DIR}")
        estudio = optuna.load_study(study_name=variante, storage=f"sqlite:///{db}")
        mejores[variante] = {"hv_f1": estudio.best_value, "parametros": estudio.best_params}
    print(f"[parámetros] reconstruidos desde los estudios .db ({PARAM_DIR})")
    return mejores

MEJORES = cargar_mejores_parametros()
for v in VARIANTES:
    print(f"  {v:<11} HV param = {MEJORES[v]['hv_f1']:.4f} | {MEJORES[v]['parametros']}")

[backend] CuPy detectado → operaciones vectorizadas en GPU.
[backend] CuPy detectado → operaciones vectorizadas en GPU.
  NSGA2       HV param = 90.2587 | {'pop_size': 174, 'crossover_prob': 0.9, 'mutation_prob': 0.04}
  NSGA2_PLS   HV param = 91.0818 | {'pop_size': 217, 'crossover_prob': 0.89, 'mutation_prob': 0.01, 'pls_neighborhood': 0.07, 'pls_local_budget_pct': 9}
  NSGA2_MOLS  HV param = 90.0614 | {'pop_size': 63, 'crossover_prob': 0.88, 'mutation_prob': 0.05, 'M_v': 'BOTH', 'mode': 'large', 'mols_local_budget_pct': 7, 'neighborhood_pct': 27}
  NSGA2_DBS   HV param = 90.3863 | {'pop_size': 266, 'crossover_prob': 0.9, 'mutation_prob': 0.05, 'mode': 'expresion', 'p_pct': 13}
  NSGA2_PNS   HV param = 91.0381 | {'pop_size': 61, 'crossover_prob': 0.8700000000000001, 'mutation_prob': 0.03, 'nF': 3}
  NSGA2_MOPR  HV param = 90.6251 | {'pop_size': 292, 'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.05}


## 3. Cuadro de ejecución

Define qué variantes, datasets y semillas se ejecutan. Los resultados quedan en `Results/Ejecuciones`:

| Archivo | Contenido |
|---|---|
| `historial_ejecuciones.csv` | Registro acumulativo (una fila por ejecución: variante, dataset, semilla, hipervolumen inicial/final, generaciones, llamadas, tiempo, fecha y parámetros). |
| `Frentes/<variante>_<dataset>_<semilla>.npz` | Frente F0 (objetivos y medoides) e historial de hipervolumen de la ejecución. |
| `seeds_ejecuciones.json` | Las 31 semillas usadas. |

Las ejecuciones ya registradas (misma variante, dataset y semilla) se omiten, salvo que `REEJECUTAR = True`; en ese caso se agrega una fila nueva al historial y el resumen usa la más reciente.

In [5]:
#######################################################
# MODULO DE EJECUCIÓN
# Ejecuta variante × dataset × semilla con los mejores hiperparámetros,
# guardando cada ejecución apenas termina (CSV de historial + .npz del frente).
#######################################################

import io                                                   # Silenciar el log por generación de los algoritmos.
import time                                                 # Tiempo por ejecución.
import contextlib                                           # redirect_stdout.
from datetime import datetime

############################
# Configuración del usuario
############################
VARIANTES_A_EJECUTAR = list(VARIANTES)                      # Subconjunto, p. ej. ["NSGA2", "NSGA2_PNS"].
DATASETS_A_EJECUTAR  = ["B1", "B2", "B3"]                   # Datasets de evaluación (prefijo B).
SEEDS_A_EJECUTAR     = SEEDS                                # Subconjunto, p. ej. SEEDS[:5] para una prueba.
REEJECUTAR           = False                                # True: repite ejecuciones ya registradas (se agrega al historial).

############################
# Constantes del experimento
############################
K                      = 4                                  # Medoides por solución.
MAX_OBJ_FUNCTION_CALLS = 100_000                            # Presupuesto POR ejecución.
HISTORIAL_FILE         = RESULTS_DIR / "historial_ejecuciones.csv"
assert set(VARIANTES_A_EJECUTAR) <= set(VARIANTES), "Variante desconocida en VARIANTES_A_EJECUTAR."

############################
# Datasets (carga bajo demanda: B puede ser muy grande)
############################
_CACHE = {}

def cargar_dataset(nombre: str) -> dict:
    """Carga GE/BI. Los M_V y sus radios se calculan solo cuando una configuración de MOLS los requiere."""
    if nombre not in _CACHE:
        _CACHE.clear()                                      # Un solo dataset en memoria a la vez.
        ge, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DE.npz")
        bi, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DB_M.npz")
        _CACHE[nombre] = {"ge": ge, "bi": bi, "n": ge.shape[0], "m_v": {}, "radios": {}}
    return _CACHE[nombre]

def radio_mols(ds: dict, fuente: str, percentil: int) -> tuple:
    """M_V y radio `neighborhood` (percentil de M_V sin diagonal), con memoización por dataset."""
    if fuente not in ds["m_v"]:
        ds["m_v"][fuente] = build_neighborhood_matrix(ds["ge"], ds["bi"], fuente)
    clave = (fuente, percentil)
    if clave not in ds["radios"]:
        m_v = ds["m_v"][fuente]
        ds["radios"][clave] = float(np.percentile(m_v[~np.eye(ds["n"], dtype=bool)], percentil))
    return ds["m_v"][fuente], ds["radios"][clave]

############################
# Argumentos de cada variante (misma traducción que en Parametrización)
############################
def kwargs_variante(variante: str, cfg: dict, ds: dict, seed: int) -> dict:
    kw = dict(
        ge_matrix=ds["ge"], bi_matrix=ds["bi"], n=ds["n"], k=K,
        pop_size=cfg["pop_size"], max_obj_function_calls=MAX_OBJ_FUNCTION_CALLS,
        crossover_prob=cfg["crossover_prob"], mutation_prob=cfg["mutation_prob"],
        seed=seed,
    )
    budget = lambda pct: MAX_OBJ_FUNCTION_CALLS * pct // 100

    if variante == "NSGA2_PLS":
        kw["pls_neighborhood"] = cfg["pls_neighborhood"]
        kw["pls_local_budget"] = budget(cfg["pls_local_budget_pct"])
    elif variante == "NSGA2_MOLS":
        m_v, radio = radio_mols(ds, cfg["M_v"], cfg["neighborhood_pct"])
        kw["mode"]              = cfg["mode"]
        kw["m_v_matrix"]        = m_v
        kw["neighborhood"]      = radio
        kw["mols_local_budget"] = budget(cfg["mols_local_budget_pct"])
    elif variante == "NSGA2_PNS":
        kw["nF"]               = cfg["nF"]
    elif variante == "NSGA2_DBS":
        kw["mode"]             = cfg["mode"]
        kw["p"]                = cfg["p_pct"] / 100
    return kw

############################
# Historial
############################
def leer_historial() -> pd.DataFrame:
    return pd.read_csv(HISTORIAL_FILE) if HISTORIAL_FILE.exists() else pd.DataFrame()

def registrar(fila: dict) -> None:
    pd.DataFrame([fila]).to_csv(HISTORIAL_FILE, mode="a", header=not HISTORIAL_FILE.exists(), index=False)

def ejecutadas() -> set:
    h = leer_historial()
    return set() if h.empty else set(zip(h["variante"], h["dataset"], h["semilla"]))

############################
# Ejecución de una corrida
############################
def ejecutar(variante: str, nombre_ds: str, semilla: int) -> dict:
    cfg = MEJORES[variante]["parametros"]
    ds  = cargar_dataset(nombre_ds)

    solution_encoding.OBJ_FUNCTION_CALLS = 0                # Reinicio obligatorio del presupuesto.
    t0 = time.time()
    with contextlib.redirect_stdout(io.StringIO()):         # Se silencia el print por generación.
        res = VARIANTES[variante](**kwargs_variante(variante, cfg, ds, semilla))
    segundos = time.time() - t0

    hv = res["history_hv_f1"]
    f0 = np.asarray(res["fronts"][0])
    np.savez_compressed(
        FRENTES_DIR / f"{variante}_{nombre_ds}_{semilla}.npz",
        objectives=np.asarray(res["objectives"])[f0],
        medoids=np.asarray(res["population"])[f0],
        history_hv_f1=np.asarray(hv, dtype=float),
    )
    fila = {
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "variante": variante, "dataset": nombre_ds, "semilla": semilla,
        "hv_inicial": float(hv[0]), "hv_final": float(hv[-1]),
        "tam_f0": int(len(f0)), "generaciones": int(res["generations_run"]),
        "llamadas": int(solution_encoding.OBJ_FUNCTION_CALLS), "segundos": round(segundos, 1),
        "parametros": json.dumps(cfg, ensure_ascii=False),
    }
    registrar(fila)
    sincronizar_drive()                                     # Respaldo en Drive tras cada ejecución (solo Colab).
    return fila

############################
# Cuadro de ejecución (dataset como bucle externo: evita recargar matrices grandes)
############################
hechas = set() if REEJECUTAR else ejecutadas()
total  = len(VARIANTES_A_EJECUTAR) * len(DATASETS_A_EJECUTAR) * len(SEEDS_A_EJECUTAR)
print(f"[plan] {total} ejecuciones en total | {len(hechas)} ya registradas")

contador = 0
for nombre_ds in DATASETS_A_EJECUTAR:
    for variante in VARIANTES_A_EJECUTAR:
        for semilla in SEEDS_A_EJECUTAR:
            contador += 1
            if (variante, nombre_ds, semilla) in hechas:
                continue
            fila = ejecutar(variante, nombre_ds, semilla)
            print(f"[{contador:>4}/{total}] {variante:<11} {nombre_ds} semilla={semilla:<11} "
                  f"HV F0 = {fila['hv_final']:.4f} | |F0| = {fila['tam_f0']:>3} | {fila['segundos'] / 60:.1f} min")
print("\n[fin] ejecuciones completadas")

[plan] 558 ejecuciones en total | 558 ya registradas

[fin] ejecuciones completadas


## 4. Resultados

Resumen del hipervolumen final de F0 por variante y dataset (usa la ejecución más reciente de cada combinación).

In [6]:
#######################################################
# MODULO DE RESUMEN DE RESULTADOS
#######################################################

historial = leer_historial()
if historial.empty:
    print("Aún no hay ejecuciones registradas.")
else:
    ultimas = historial.sort_values("fecha").drop_duplicates(["variante", "dataset", "semilla"], keep="last")

    resumen = (ultimas.groupby(["dataset", "variante"])
               .agg(n=("hv_final", "size"), hv_media=("hv_final", "mean"), hv_desv=("hv_final", "std"),
                    hv_min=("hv_final", "min"), hv_max=("hv_final", "max"),
                    tam_f0_medio=("tam_f0", "mean"), minutos_medio=("segundos", lambda s: s.mean() / 60))
               .round(4))
    resumen.to_csv(RESULTS_DIR / "resumen_resultados.csv")
    sincronizar_drive()
    display(resumen)

    # Matriz semilla × variante (HV final) por dataset: base para las pruebas estadísticas.
    for nombre_ds, g in ultimas.groupby("dataset"):
        print(f"\n{nombre_ds}: HV final por semilla")
        display(g.pivot(index="semilla", columns="variante", values="hv_final").round(4))

n  hv_media  hv_desv   hv_min   hv_max  tam_f0_medio  \
dataset variante                                                            
B1      NSGA2       31   88.3836   0.3493  87.6443  88.8363       11.2258   
        NSGA2_DBS   31   88.9281   0.2865  88.3792  89.4619       11.4516   
        NSGA2_MOLS  31   88.5598   0.3873  87.9121  89.1758       21.0645   
        NSGA2_MOPR  31   88.7755   0.3435  87.8346  89.3967       13.5484   
        NSGA2_PLS   31   89.3951   0.1313  89.0353  89.5693       16.0323   
        NSGA2_PNS   31   89.4165   0.1522  89.0747  89.6616       13.5161   
B2      NSGA2       31   90.2278   0.2195  89.7755  90.6090       11.7419   
        NSGA2_DBS   31   90.5733   0.1997  90.1707  91.0002       11.3226   
        NSGA2_MOLS  31   90.1815   0.3419  89.6067  90.6888       20.8710   
        NSGA2_MOPR  31   90.4720   0.2007  90.1764  90.9315       15.3226   
        NSGA2_PLS   31   90.9214   0.1096  90.7195  91.2295       21.0645   
        NSGA2_PNS   31   90.7481   0.2287  90.0151  91.1606       18.0968   
B3      NSGA2       31   88.7192   0.1769  88.4380  89.0068       22.3226   
        NSGA2_DBS   31   88.9655   0.0998  88.7601  89.1578       15.2903   
        NSGA2_MOLS  31   88.8924   0.1936  88.4932  89.1660       24.8387   
        NSGA2_MOPR  31   88.7896   0.1760  88.4059  89.0854       25.6452   
        NSGA2_PLS   31   89.2224   0.1533  88.8891  89.5508       26.4194   
        NSGA2_PNS   31   89.2310   0.1130  88.9908  89.4122       32.7742   

                    minutos_medio  
dataset variante                   
B1      NSGA2              0.3801  
        NSGA2_DBS          0.2315  
        NSGA2_MOLS         0.0559  
        NSGA2_MOPR         1.2071  
        NSGA2_PLS          0.1276  
        NSGA2_PNS          0.3213  
B2      NSGA2              0.3711  
        NSGA2_DBS          0.2276  
        NSGA2_MOLS         0.0640  
        NSGA2_MOPR         1.1723  
        NSGA2_PLS          0.1206  
        NSGA2_PNS          0.3174  
B3      NSGA2              1.4954  
        NSGA2_DBS          0.5895  
        NSGA2_MOLS         0.1892  
        NSGA2_MOPR        16.7140  
        NSGA2_PLS          0.4487  
        NSGA2_PNS          2.9880


B1: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,88.3417,88.7210,88.1873,88.5515,89.2817,89.3164
149968717,88.8363,89.0216,89.1521,89.1665,89.3915,89.2568
245761548,88.6408,89.1614,88.4191,88.8122,89.4188,89.3227
291931378,88.4533,88.7403,88.1098,88.3968,89.4435,89.5604
346868666,87.7009,88.9811,88.2724,88.2911,89.1701,89.6616
358181644,88.7838,88.6471,88.5586,88.1950,89.3638,89.2972
398383450,88.7474,88.9892,88.9037,88.9323,89.4286,89.6501
536623952,88.0079,88.8207,88.2039,88.7434,89.4175,89.5289
570232917,88.5605,88.6250,88.0046,88.8826,89.5481,89.0747



B2: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,90.5193,90.5589,90.0127,90.3979,90.9490,90.6255
149968717,90.4648,90.5487,90.5431,90.3109,91.0186,91.0046
245761548,90.5536,90.8733,90.4593,90.5445,90.9315,90.6557
291931378,90.1353,90.4411,90.6371,90.4224,90.8983,90.3369
346868666,90.5846,90.8455,89.7201,90.2408,90.8724,91.1606
358181644,90.6090,90.5421,89.9338,90.5824,90.9054,90.7136
398383450,90.0781,90.4019,89.9705,90.4622,90.9372,90.8418
536623952,89.9648,90.4519,90.6888,90.5305,91.0138,90.9957
570232917,89.9180,90.7774,90.3946,90.1959,90.7654,90.6093



B3: HV final por semilla


variante,NSGA2,NSGA2_DBS,NSGA2_MOLS,NSGA2_MOPR,NSGA2_PLS,NSGA2_PNS
semilla,,,,,,
2241395,88.8131,88.9400,89.0154,88.6329,89.2262,89.3997
149968717,88.5431,88.9005,89.0411,88.8551,88.9765,89.3554
245761548,88.5689,89.0887,89.0852,89.0746,88.9884,89.1328
291931378,88.5123,88.9993,88.8218,88.7140,89.1388,89.3014
346868666,88.8694,88.9815,88.8464,88.6875,89.3369,89.0728
358181644,88.9486,88.8899,88.5404,88.9197,89.2824,89.2518
398383450,88.8615,89.0470,89.0301,88.7424,89.1661,89.1729
536623952,88.5885,89.0472,88.6194,88.9765,89.2743,89.1608
570232917,88.8220,89.0309,88.8145,88.4059,89.3508,89.3287
